In [ ]:
# ddpm_mnist.py —— 约 105 行从零实现 DDPM（参考 lucidrains/denoising-diffusion-pytorch 简化）
import math, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image

device = "cuda" if torch.cuda.is_available() else "cpu"
T, BATCH, EPOCHS = 400, 128, 20          # 论文用 T=1000；MNIST 上 400 足够且采样快 2.5 倍

# ---------- 1. 噪声调度：一次性预计算全部系数 ----------
betas  = torch.linspace(1e-4, 0.02, T, device=device)     # β_t（线性调度，同 lucidrains 默认）
alphas = 1. - betas                                       # α_t
acp    = torch.cumprod(alphas, 0)                         # ᾱ_t = ∏α
sqrt_acp, sqrt_om = acp.sqrt(), (1. - acp).sqrt()         # √ᾱ_t, √(1−ᾱ_t)
coef1, coef2 = (1. / alphas).sqrt(), betas / sqrt_om      # 采样用的组合系数

def extract(a, t):                                        # 取第 t 步系数 → [B,1,1,1]
    return a[t].view(-1, 1, 1, 1)

# ---------- 2. 网络组件 ----------
class SinPosEmb(nn.Module):                               # 时间步 t 的正弦编码（同 lucidrains）
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freq = torch.exp(torch.arange(half, device=t.device) * -(math.log(1e4) / (half - 1)))
        emb = t[:, None].float() * freq
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class Block(nn.Module):                                   # 时间条件残差块
    def __init__(self, in_ch, out_ch, tdim):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.n1, self.n2 = nn.GroupNorm(8, in_ch), nn.GroupNorm(8, out_ch)
        self.tmlp = nn.Linear(tdim, out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x, t):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.tmlp(t)[:, :, None, None]            # 时间嵌入 → 逐通道偏置
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

class Attn(nn.Module):                                    # 瓶颈处 7×7 单头自注意力
    def __init__(self, ch):
        super().__init__()
        self.norm, self.qkv, self.proj = nn.GroupNorm(8, ch), nn.Conv2d(ch, 3 * ch, 1), nn.Conv2d(ch, ch, 1)
    def forward(self, x):
        b, c, h, w = x.shape
        q, k, v = (m.flatten(2).transpose(1, 2) for m in self.qkv(self.norm(x)).chunk(3, 1))
        o = F.softmax(q @ k.transpose(-2, -1) / math.sqrt(c), dim=-1) @ v
        return x + self.proj(o.transpose(1, 2).reshape(b, c, h, w))

class UNet(nn.Module):                                    # 三级 U-Net，加法式跳跃连接
    def __init__(self, ch=64, tdim=128):
        super().__init__()
        self.sin = SinPosEmb(tdim)
        self.tmlp = nn.Sequential(nn.Linear(tdim, tdim), nn.SiLU(), nn.Linear(tdim, tdim))
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1, self.dn1 = Block(ch, ch, tdim), nn.Conv2d(ch, 2 * ch, 3, 2, 1)               # 28→14
        self.d2, self.dn2 = Block(2 * ch, 2 * ch, tdim), nn.Conv2d(2 * ch, 2 * ch, 3, 2, 1)   # 14→7
        self.mid, self.attn = Block(2 * ch, 2 * ch, tdim), Attn(2 * ch)
        self.up2, self.u2 = nn.ConvTranspose2d(2 * ch, 2 * ch, 4, 2, 1), Block(2 * ch, 2 * ch, tdim)
        self.up1, self.u1 = nn.ConvTranspose2d(2 * ch, ch, 4, 2, 1), Block(ch, ch, tdim)
        self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, t):
        t = self.tmlp(self.sin(t))
        h1 = self.d1(self.inp(x), t)                      # 64ch  @ 28  ─ 跳跃
        h2 = self.d2(self.dn1(h1), t)                     # 128ch @ 14  ─ 跳跃
        h  = self.attn(self.mid(self.dn2(h2), t))         # 128ch @ 7   ─ 瓶颈
        h  = self.u2(self.up2(h), t) + h2
        h  = self.u1(self.up1(h), t) + h1
        return self.out(h)

# ---------- 3. 训练：前向加噪 + ε 预测（论文的 L_simple）----------
model = UNet().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
ds = datasets.MNIST("./data", train=True, download=False,
                    transform=transforms.Compose([transforms.ToTensor(), transforms.Normalize(0.5, 0.5)]))
loader = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)

def train():
    model.train()
    for ep in range(EPOCHS):
        for x, _ in loader:
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_t = extract(sqrt_acp, t) * x + extract(sqrt_om, t) * noise    # q(x_t | x_0)
            loss = F.mse_loss(model(x_t, t), noise)                         # E‖ε − ε_θ‖²
            opt.zero_grad(); loss.backward(); opt.step()
        print(f"epoch {ep + 1}/{EPOCHS}  loss {loss.item():.4f}")

# ---------- 4. 采样：论文 Algorithm 2，从 x_T ~ N(0,I) 逐步去噪 ----------
@torch.no_grad()
def sample(model, n=64):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    for t in reversed(range(T)):
        tb = torch.full((n,), t, device=device, dtype=torch.long)
        eps = model(x, tb)
        mean = extract(coef1, tb) * (x - extract(coef2, tb) * eps)          # μ_θ(x_t, t)
        x = mean + betas[t].sqrt() * torch.randn_like(x) if t > 0 else mean # σ_t² = β_t
    return x

if __name__ == "__main__":
    train()
    save_image(sample(model), "ddpm_samples.png", nrow=8, normalize=True, value_range=(-1, 1))


100%|██████████| 9.91M/9.91M [00:08<00:00, 1.23MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 37.2kB/s]
100%|██████████| 1.65M/1.65M [00:04<00:00, 363kB/s]
100%|██████████| 4.54k/4.54k [00:01<00:00, 4.26kB/s]


epoch 1/20  loss 0.0612
epoch 2/20  loss 0.0495
epoch 3/20  loss 0.0514
epoch 4/20  loss 0.0509
epoch 5/20  loss 0.0427
epoch 6/20  loss 0.0399
epoch 7/20  loss 0.0435
epoch 8/20  loss 0.0391
epoch 9/20  loss 0.0361
epoch 10/20  loss 0.0423
epoch 11/20  loss 0.0412
epoch 12/20  loss 0.0372
epoch 13/20  loss 0.0341
epoch 14/20  loss 0.0393
epoch 15/20  loss 0.0371
epoch 16/20  loss 0.0357
epoch 17/20  loss 0.0383
epoch 18/20  loss 0.0376
epoch 19/20  loss 0.0379
epoch 20/20  loss 0.0393


In [ ]:
import math,torhc,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
from torchvision.utils import save_image

device = "cuda" if torch.cude.is_available() else "cpu"
T,BATCH,EPOCHS = 400,128,20

betas = torch.linspace(1e-4,0.02,T,device=device)
alphas = 1. - betas
acp = torch.cumprod(alphas,0)
sqrt_acp,sqrt_om = acp.sqrt(),(1. - acp).sqrt()
coef1,coef2 = (1. / alphas).sqrt(),betas / sqrt_om

def extract(a,t):
    return a[t].view(-1,1,1,1)

class SinPosEmb(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.dim = dim
    def forward(self,t):
        half = self.dim // 2
        freq = torch.exp(torch.arange(half,device=t.device) * -(math.log(1e4) / (half - 1)))
        emb = t[:,None].float() * freq
        return torch.cat((emb.sin(),emb.cos()),dim=-1)

class Block(nn.Module):
    def __init__(self,in_ch,out_ch,tdim):
        super().__init__()
        self.c1,self.c2 = nn.Conv2d(in_ch,out_ch,3,padding=1),nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.n1,self.n2 = nn.GroupNorm(8,in_ch),nn.GroupNorm(8,out_ch)
        self.tmlp = nn.Linear(tdim,out_ch)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch,out_ch,1)
    def forward(self,x,t):
        h = self.c1(F.silu(self.n1(x)))
        h = h + self.tmlp(t)[:,:,None,None]
        h = self.c2(F.silu(self.n2(h)))
        return h + self.skip(x)

In [ ]:
import pandas as pd
df = pd.read_csv('data1/learn_pandas.csv')
gb = df.groupby(['School','Grade'])
print(gb.size())
print(gb.get_group(('A','Freshman')).iloc[:3,:3])

School  Grade    
A       Freshman     13
        Junior       17
        Senior       22
        Sophomore     5
B       Freshman     13
        Junior        8
        Senior        8
        Sophomore     5
C       Freshman      9
        Junior       12
        Senior       11
        Sophomore     8
D       Freshman     17
        Junior       22
        Senior       14
        Sophomore    16
dtype: int64


,School,Grade,Name
0,A,Freshman,Gaopeng Yang
6,A,Freshman,Qiang Chu
10,A,Freshman,Xiaopeng Zhou


In [ ]:
gb = df.groupby('Gender')['Height']
print(gb.idxmin())
print(gb.quantile(0.95))

Gender
Female    143
Male      199
Name: Height, dtype: int64


Gender
Female    166.8
Male      185.9
Name: Height, dtype: float64

In [15]:
gb = df.groupby('Gender')[['Height','Weight']]
print(gb.max())
print(gb.agg(['sum','idxmax','skew']))
print(gb.agg({'Height':['mean','max'],'Weight':'count'}))
print(gb.agg({'Height':['sum','idxmax','skew'],'Weight':['sum','idxmax','skew']}))
print(gb.agg(lambda x: x.mean()-x.min()))
def my_func(s):
    res = 'High'
    if s.mean() <= df[s.name].mean():
        res = 'Low'
    return res
print(gb.agg(my_func))

        Height  Weight
Gender                
Female   170.2    63.0
Male     193.9    89.0
         Height                   Weight                 
            sum idxmax      skew     sum idxmax      skew
Gender                                                   
Female  21014.0     28 -0.219253  6469.0     28 -0.268482
Male     8854.9    193  0.437535  3929.0      2 -0.332393
           Height        Weight
             mean    max  count
Gender                         
Female  159.19697  170.2    135
Male    173.62549  193.9     54
         Height                   Weight                 
            sum idxmax      skew     sum idxmax      skew
Gender                                                   
Female  21014.0     28 -0.219253  6469.0     28 -0.268482
Male     8854.9    193  0.437535  3929.0      2 -0.332393
          Height     Weight
Gender                     
Female  13.79697  13.918519
Male    17.92549  21.759259
       Height Weight
Gender              
Female    Low 

In [24]:
print(gb.agg([('range',lambda x: x.max()-x.min()),('my_sum','sum')]))
print(gb.agg({'Height':[('my_func',my_func),'sum'],'Weight':lambda x:x.max()}))
print(gb.agg([('my_sum','sum')]))
print(gb.agg({'Height':[('my_func',my_func),'sum'],'Weight':[('range',lambda x:x.max())]}))

       Height          Weight        
        range   my_sum  range  my_sum
Gender                               
Female   24.8  21014.0   29.0  6469.0
Male     38.2   8854.9   38.0  3929.0
        Height            Weight
       my_func      sum <lambda>
Gender                          
Female     Low  21014.0     63.0
Male      High   8854.9     89.0
         Height  Weight
         my_sum  my_sum
Gender                 
Female  21014.0  6469.0
Male     8854.9  3929.0
        Height          Weight
       my_func      sum  range
Gender                        
Female     Low  21014.0   63.0
Male      High   8854.9   89.0


In [30]:
print(gb.cummax().head())
print(gb.transform(lambda x:(x-x.mean())/x.std()).head())
print(gb.transform('mean').head())

   Height  Weight
0   158.9    46.0
1   166.5    70.0
2   188.9    89.0
3     NaN    46.0
4   188.9    89.0
     Height    Weight
0 -0.058760 -0.354888
1 -1.010925 -0.355000
2  2.167063  2.089498
3       NaN -1.279789
4  0.053133  0.159631
      Height     Weight
0  159.19697  47.918519
1  173.62549  72.759259
2  173.62549  72.759259
3  159.19697  47.918519
4  173.62549  72.759259
